# Post-Pandemic Gender Labor Force Trends
Principle Investigators: Abigail Garvey, Anna Wang, Rachelle Lang - Group Q


In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

The way this data is organized is many series that examine different statistics about the labor force through different demographic characteristics, such as industry, age, gender, remote work, veteran status, and many more. ln.series is a file containing descriptions of all the series and which variables they address. All_data contains every data point for each series, labelled by its series id.

In [2]:
# a list of all the series contained in all_data
all_series = pd.read_csv('../BLS/ln.series.txt', sep='\t')
all_series.columns = all_series.columns.str.strip()
all_series = all_series.set_index('series_id')
all_series.index = all_series.index.str.strip()

# identified by its series, a date in that series, and the value
all_data = pd.read_csv('../BLS/ln.data.1.AllData.txt', sep='\t')
all_data.columns = all_data.columns.str.strip()
all_data = all_data.set_index('series_id')
all_data.index = all_data.index.str.strip()
all_data = all_data[all_data['period'] != 'M13'] # according to ChatGPT it's the average

# all the codes like age, education, gender, that distinguish a series
codes = all_series.filter(regex='.*_code$').columns.drop(['lfst_code', 'periodicity_code', 'pcts_code', 'tdat_code'])

C:\Users\famil\AppData\Local\Temp\ipykernel_29412\3059707570.py:8: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  all_data = pd.read_csv('../BLS/ln.data.1.AllData.txt', sep='\t')


Generate industries.csv: the number of people of each gender employed in each occupation/industry across time

In [3]:
# find all series about industry and occupation by gender

irrelevant_codes = codes.drop(['sexs_code', 'occupation_code', 'indy_code'])

def industry_gender(gender):
    # filter by gender
    industry_g = all_series[all_series['sexs_code'] == gender]
    # make sure series tracks industry and/or occupation
    industry_g = industry_g[(industry_g['occupation_code'] != 0) |
                            (industry_g['indy_code'] != 0)]
    # make sure no other code we're not interested in gets into the series
    for c in irrelevant_codes:
        industry_g = industry_g[industry_g[c] == 0]
    # measure just normal employment count, not some percentage
    industry_g = industry_g[industry_g['pcts_code'] == 0]

    return industry_g

to_drop = list(irrelevant_codes) + ['lfst_code', 'pcts_code', 'tdat_code', 'footnote_codes']
im_series = industry_gender(1).drop(columns=to_drop)
iw_series = industry_gender(2).drop(columns=to_drop)

In [4]:
# merge series with data

im_data = all_data.loc[im_series.index]
iw_data = all_data.loc[iw_series.index]
im_data = pd.merge(im_data, im_series.drop(columns=['begin_year', 'begin_period', 'end_year', 'end_period']), on='series_id')
im_data = im_data.reset_index('series_id')
iw_data = pd.merge(iw_data, iw_series.drop(columns=['begin_year', 'begin_period', 'end_year', 'end_period']), on='series_id')
iw_data = iw_data.reset_index('series_id')

In [5]:
# convert to datetime objects

def make_datetime(row):
    p_code = row['period'][0]
    if p_code == 'M':
        return pd.to_datetime(f'{row["year"]}/{row["period"][1:]}/01', format='%Y/%m/%d')
    if p_code == 'Q':
        return pd.to_datetime(f'{row["year"]}/{int(row["period"][1:])*3}/01', format='%Y/%m/%d') # not sure where in the quarter it should correspond too, but maybe doesn't matter
    print(p_code)

im_data['date'] = im_data.apply(make_datetime, axis=1)
im_data = im_data.set_index(['indy_code', 'occupation_code', 'date', 'series_id'])
im_data = im_data.sort_index()

iw_data['date'] = iw_data.apply(make_datetime, axis=1)
iw_data = iw_data.set_index(['indy_code', 'occupation_code', 'date', 'series_id'])
iw_data = iw_data.sort_index()
iw_data

year period    value  \
indy_code occupation_code date       series_id                            
0         7               1983-01-01 LNU02032526   1983    M01  12123.0   
                          1983-02-01 LNU02032526   1983    M02  12225.0   
                          1983-03-01 LNU02032526   1983    M03  12201.0   
                          1983-04-01 LNU02032526   1983    M04  12366.0   
                          1983-05-01 LNU02032526   1983    M05  12367.0   
...                                                 ...    ...      ...   
9369      8999            2025-05-01 LNU02042290   2025    M05       15   
                          2025-06-01 LNU02042290   2025    M06        7   
                                     LNU02042290Q  2025    Q02       15   
                          2025-07-01 LNU02042290   2025    M07       11   
                          2025-08-01 LNU02042290   2025    M08       18   

                                                  footnote_codes  \
indy_code occupation_code date       series_id                     
0         7               1983-01-01 LNU02032526               3   
                          1983-02-01 LNU02032526             NaN   
                          1983-03-01 LNU02032526             NaN   
                          1983-04-01 LNU02032526             NaN   
                          1983-05-01 LNU02032526             NaN   
...                                                          ...   
9369      8999            2025-05-01 LNU02042290             NaN   
                          2025-06-01 LNU02042290             NaN   
                                     LNU02042290Q            NaN   
                          2025-07-01 LNU02042290             NaN   
                          2025-08-01 LNU02042290             NaN   

                                                  periodicity_code  \
indy_code occupation_code date       series_id                       
0         7               1983-01-01 LNU02032526                 M   
                          1983-02-01 LNU02032526                 M   
                          1983-03-01 LNU02032526                 M   
                          1983-04-01 LNU02032526                 M   
                          1983-05-01 LNU02032526                 M   
...                                                            ...   
9369      8999            2025-05-01 LNU02042290                 M   
                          2025-06-01 LNU02042290                 M   
                                     LNU02042290Q                Q   
                          2025-07-01 LNU02042290                 M   
                          2025-08-01 LNU02042290                 M   

                                                                                        series_title  \
indy_code occupation_code date       series_id                                                         
0         7               1983-01-01 LNU02032526   (Unadj) Employment Level - Management, Profess...   
                          1983-02-01 LNU02032526   (Unadj) Employment Level - Management, Profess...   
                          1983-03-01 LNU02032526   (Unadj) Employment Level - Management, Profess...   
                          1983-04-01 LNU02032526   (Unadj) Employment Level - Management, Profess...   
                          1983-05-01 LNU02032526   (Unadj) Employment Level - Management, Profess...   
...                                                                                              ...   
9369      8999            2025-05-01 LNU02042290   (unadj) Employed - Public administration, Tran...   
                          2025-06-01 LNU02042290   (unadj) Employed - Public administration, Tran...   
                                     LNU02042290Q  (unadj) Employed - Public administration, Tran...   
                          2025-07-01 LNU02042290   (unadj) Employed - Public administration, Tran...   
                          2025-

In [6]:
# handle duplicates by only keeping the month level
im_data = im_data.loc[im_data.groupby(level = ['indy_code', 'occupation_code', 'date'])['periodicity_code'].idxmin()]
im_data = im_data.reset_index('series_id')
iw_data = iw_data.loc[iw_data.groupby(level = ['indy_code', 'occupation_code', 'date'])['periodicity_code'].idxmin()]
iw_data = iw_data.reset_index('series_id')

In [7]:
# merge men and women
# it just so happens that every datapoint for men has a corresponding datapoint for women
ig_data_unclean = pd.merge(im_data, iw_data, how='outer', left_index=True, right_index=True, indicator=True, suffixes=['_m', '_w'])
ig_data = ig_data_unclean[['value_m', 'value_w']]
ig_data = ig_data.rename({'value_m': 'employed_men', 'value_w': 'employed_women'}, axis=1)
ig_data['employed_total'] = ig_data['employed_men'] + ig_data['employed_women']
ig_data['pct_men'] = ig_data['employed_men'] / ig_data['employed_total'].replace(0, np.nan)
ig_data['pct_women'] = ig_data['employed_women'] / ig_data['employed_total'].replace(0, np.nan)
ig_data

C:\Users\famil\AppData\Local\Temp\ipykernel_29412\3408799488.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ig_data['pct_men'] = ig_data['employed_men'] / ig_data['employed_total'].replace(0, np.nan)
C:\Users\famil\AppData\Local\Temp\ipykernel_29412\3408799488.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ig_data['pct_women'] = ig_data['employed_women'] / ig_data['employed_total'].replace(0, np.nan)


employed_men employed_women  \
indy_code occupation_code date                                     
0         7               1983-01-01      16410.0        12123.0   
                          1983-02-01      16355.0        12225.0   
                          1983-03-01      16491.0        12201.0   
                          1983-04-01      16759.0        12366.0   
                          1983-05-01      16825.0        12367.0   
...                                           ...            ...   
9369      8999            2025-04-01          147             22   
                          2025-05-01          130             15   
                          2025-06-01          141              7   
                          2025-07-01          166             11   
                          2025-08-01          146             18   

                                     employed_total   pct_men pct_women  
indy_code occupation_code date                                           
0         7               1983-01-01        28533.0  0.575124  0.424876  
                          1983-02-01        28580.0  0.572253  0.427747  
                          1983-03-01        28692.0   0.57476   0.42524  
                          1983-04-01        29125.0  0.575416  0.424584  
                          1983-05-01        29192.0  0.576357  0.423643  
...                                             ...       ...       ...  
9369      8999            2025-04-01            169  0.869822  0.130178  
                          2025-05-01            145  0.896552  0.103448  
                          2025-06-01            148  0.952703  0.047297  
                          2025-07-01            177  0.937853  0.062147  
                          2025-08-01            164  0.890244  0.109756  

[71528 rows x 5 columns]

In [8]:
ig_data.to_csv('../Data/industries.csv') # save to file

Generate lfpr.csv and lfpr_ages.csv: labor force participation rates by gender, and by gender and age

In [9]:
# get series codes for labor force participation rates of men and women

lfp_rates  = all_series[(all_series['lfst_code'] == 13) &
           (all_series['sexs_code'] != 0)
           ]
irrelevant_codes = codes.drop(['sexs_code', 'ages_code'])
for c in irrelevant_codes:
        lfp_rates = lfp_rates[lfp_rates[c] == 0]
lfp_rates = lfp_rates[lfp_rates['periodicity_code'] == 'M'] # more datapoints
lfp_rates = lfp_rates[lfp_rates['seasonal'] == 'S'] # seasonally adjusted seems more accurate?
lfp_id_m = lfp_rates.index[0]
lfp_id_w = lfp_rates.index[1]
lfp_rates.head()

,lfst_code,periodicity_code,series_title,absn_code,activity_code,ages_code,cert_code,class_code,duration_code,education_code,...,born_code,chld_code,disa_code,seasonal,tlwk_code,footnote_codes,begin_year,begin_period,end_year,end_period
series_id,,,,,,,,,,,,,,,,,,,,,
LNS11300001,13,M,(Seas) Labor Force Participation Rate - Men,0,0,0,0,0,0,0,...,0,0,0,S,0,NaN,1948,M01,2025,M08
LNS11300002,13,M,(Seas) Labor Force Participation Rate - Women,0,0,0,0,0,0,0,...,0,0,0,S,0,NaN,1948,M01,2025,M08
LNS11300013,13,M,(Seas) Labor Force Participation Rate - 16-19 ...,0,0,8,0,0,0,0,...,0,0,0,S,0,NaN,1948,M01,2025,M08
LNS11300014,13,M,(Seas) Labor Force Participation Rate - 16-19 ...,0,0,8,0,0,0,0,...,0,0,0,S,0,NaN,1948,M01,2025,M08
LNS11300025,13,M,(Seas) Labor Force Participation Rate - 20 yrs...,0,0,17,0,0,0,0,...,0,0,0,S,0,NaN,1948,M01,2025,M08


In [10]:
# get series
lfprm = all_data.loc[lfp_id_m]
lfprw = all_data.loc[lfp_id_w]
lfprm['date'] = lfprm.apply(make_datetime, axis=1)
lfprw['date'] = lfprw.apply(make_datetime, axis=1)
lfpr = pd.merge(lfprm.drop(['period', 'footnote_codes', 'year'], axis=1), lfprw.drop(['period', 'footnote_codes', 'year'], axis=1), how='outer', on=['date'], suffixes=['_m', '_w'])
lfpr = lfpr.set_index('date')
lfpr = lfpr.rename({'value_m': 'men', 'value_w': 'women'}, axis=1)
lfpr

C:\Users\famil\AppData\Local\Temp\ipykernel_29412\2314169925.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lfprm['date'] = lfprm.apply(make_datetime, axis=1)
C:\Users\famil\AppData\Local\Temp\ipykernel_29412\2314169925.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lfprw['date'] = lfprw.apply(make_datetime, axis=1)


,men,women
date,,
1948-01-01,86.7,32.0
1948-02-01,87.0,32.4
1948-03-01,86.3,32.1
1948-04-01,86.6,33.0
1948-05-01,86.1,32.0
...,...,...
2025-04-01,68.1,57.5
2025-05-01,67.8,57.2
2025-06-01,67.8,57.0


In [11]:
# get age series
lfprm = all_data.loc[lfp_id_m]
lfprw = all_data.loc[lfp_id_w]
lfprm['date'] = lfprm.apply(make_datetime, axis=1)
lfprw['date'] = lfprw.apply(make_datetime, axis=1)
lfpr = pd.merge(lfprm.drop(['period', 'footnote_codes', 'year'], axis=1), lfprw.drop(['period', 'footnote_codes', 'year'], axis=1), how='outer', on=['date'], suffixes=['_m', '_w'])
lfpr = lfpr.set_index('date')
lfpr = lfpr.rename({'value_m': 'men', 'value_w': 'women'}, axis=1)
lfpr

C:\Users\famil\AppData\Local\Temp\ipykernel_29412\150174713.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lfprm['date'] = lfprm.apply(make_datetime, axis=1)
C:\Users\famil\AppData\Local\Temp\ipykernel_29412\150174713.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lfprw['date'] = lfprw.apply(make_datetime, axis=1)


,men,women
date,,
1948-01-01,86.7,32.0
1948-02-01,87.0,32.4
1948-03-01,86.3,32.1
1948-04-01,86.6,33.0
1948-05-01,86.1,32.0
...,...,...
2025-04-01,68.1,57.5
2025-05-01,67.8,57.2
2025-06-01,67.8,57.0


In [12]:
lfpr_ages = pd.DataFrame()
ages_labels = ['16-24', '25-34', '35-44', '45-54', '55+']
ages_codes = [10, 31, 38, 42, 45]
sexs_codes = [1, 2]
sexs_labels = [0, 1]
for c in range(len(ages_codes)):
    for s in range(len(sexs_codes)):
        series_ids = lfp_rates[(lfp_rates['ages_code'] == ages_codes[c]) & (lfp_rates['sexs_code'] == sexs_codes[s])].index
        new_data = all_data[all_data.index.isin(series_ids)]
        new_data['age'] = ages_labels[c]
        new_data['female'] = sexs_labels[s]
        lfpr_ages = pd.concat([new_data, lfpr_ages])

lfpr_ages['date'] = lfpr_ages.apply(make_datetime, axis=1)
lfpr_ages = lfpr_ages.set_index('date')
# lfpr = lfpr[['value_m', 'value_w']]
lfpr_ages = lfpr_ages.drop(['period', 'footnote_codes', 'year'], axis=1)
lfpr_ages

C:\Users\famil\AppData\Local\Temp\ipykernel_29412\2792701930.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['age'] = ages_labels[c]
C:\Users\famil\AppData\Local\Temp\ipykernel_29412\2792701930.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_data['female'] = sexs_labels[s]
C:\Users\famil\AppData\Local\Temp\ipykernel_29412\2792701930.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

,value,age,female
date,,,
1948-01-01,16.2,55+,1
1948-02-01,16.5,55+,1
1948-03-01,17.3,55+,1
1948-04-01,17.6,55+,1
1948-05-01,17.3,55+,1
...,...,...,...
2025-04-01,57.3,16-24,0
2025-05-01,56.4,16-24,0
2025-06-01,56.4,16-24,0


In [13]:
lfpr_ages.to_csv('../Data/lfpr_ages.csv')

In [14]:
lfpr.to_csv('../Data/lfpr.csv')